# Full diffusion-ready research graph

This notebook constructs the full induced directed weighted graph for users present in both raw datasets. The topology is static because `network_edges.csv` has no timestamps; only tweet/content node features evolve over monthly snapshots. No 10K subset, diffusion implementation, or EvolveGCN model is created here.

## 1. Load complete raw data and define the research population

In [ ]:
from pathlib import Path
import json
import math

import networkx as nx
from networkx.algorithms.link_analysis.pagerank_alg import _pagerank_python
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from textblob import TextBlob

candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next((p for p in candidates if (p / 'data' / 'raw' / 'master_tweets.csv').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from the project root or its notebooks directory.')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
TABLES_DIR = PROJECT_ROOT / 'results' / 'tables'
FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
for directory in [PROCESSED_DIR, TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Raw files are only read. All derived state below is held in memory or written to processed/results paths.
tweets = pd.read_csv(RAW_DIR / 'master_tweets.csv')
edges = pd.read_csv(RAW_DIR / 'network_edges.csv')
tweet_users = set(tweets['User'].astype(str))
network_nodes = set(edges['Source'].astype(str)) | set(edges['Target'].astype(str))
overlap_users = tweet_users & network_nodes
print(f'Overlapping research users: {len(overlap_users):,}')

def save_bar_chart(labels, values, title, filename, color=(55,126,184)):
    width, height, margin = 1200, 680, 85
    image = Image.new('RGB', (width,height), 'white')
    draw, font = ImageDraw.Draw(image), ImageFont.load_default()
    maximum = max(values) if values else 1
    plot_width, plot_height = width-2*margin, height-2*margin
    draw.text((margin,25), title, fill='black', font=font)
    draw.line((margin,height-margin,width-margin,height-margin), fill='black', width=2)
    draw.line((margin,margin,margin,height-margin), fill='black', width=2)
    slot = plot_width / max(len(values), 1)
    bar_width = max(2, int(slot*0.70))
    for i, (label,value) in enumerate(zip(labels,values)):
        x0 = int(margin+i*slot+(slot-bar_width)/2)
        y0 = int(height-margin-(value/maximum)*(plot_height-45)) if maximum else height-margin
        draw.rectangle((x0,y0,x0+bar_width,height-margin),fill=color)
        if len(values) <= 30:
            draw.text((x0,max(margin+22,y0-15)),f'{value:.3g}',fill='black',font=font)
            draw.text((x0,height-margin+8),str(label)[:18],fill='black',font=font)
    output = FIGURES_DIR / filename
    image.save(output)
    print(f'Saved figure: {output.relative_to(PROJECT_ROOT)}')

def save_histogram(values, title, filename, bins=40, log1p=False):
    series = pd.Series(values).dropna().astype(float)
    if log1p:
        series = series.map(math.log1p)
    lower, upper = series.min(), series.max()
    if lower == upper:
        counts, labels = [len(series)], [f'{lower:.2f}']
    else:
        step = (upper-lower)/bins
        counts = [0]*bins
        for value in series:
            counts[min(int((value-lower)/step),bins-1)] += 1
        labels = [f'{lower+i*step:.2f}' for i in range(bins)]
    save_bar_chart(labels,counts,title,filename,color=(77,175,74))


## 2. Construct the static full induced graph

In [ ]:
# Keep only edges whose endpoints are both in the research population. This is an induced directed weighted graph.
induced_edges = edges.loc[edges['Source'].isin(overlap_users) & edges['Target'].isin(overlap_users)].copy()
G = nx.DiGraph()
G.add_nodes_from(sorted(overlap_users))  # preserves overlap users even if isolated after induction
G.add_weighted_edges_from(induced_edges[['Source','Target','Weight']].itertuples(index=False, name=None), weight='Weight')
weak_components = list(nx.weakly_connected_components(G))
strong_components = list(nx.strongly_connected_components(G))
isolated_users = list(nx.isolates(G))
self_loops = list(nx.selfloop_edges(G))
graph_summary = {
    'overlapping_users': len(overlap_users), 'induced_edges': G.number_of_edges(), 'graph_density': nx.density(G),
    'weakly_connected_components': len(weak_components), 'largest_weak_component_size': max(map(len,weak_components)),
    'strongly_connected_components': len(strong_components), 'largest_strong_component_size': max(map(len,strong_components)),
    'isolated_users': len(isolated_users), 'self_loops': len(self_loops)
}
print(json.dumps(graph_summary,indent=2))
print('Self-loops are retained. They are static edge records and should be assessed by sensitivity analysis before later diffusion modelling.')
induced_edge_path = PROCESSED_DIR / 'full_overlap_induced_edges.csv'
induced_edges.to_csv(induced_edge_path,index=False)
print(f'Saved {induced_edge_path.relative_to(PROJECT_ROOT)}')

## 3. Static network features: degrees, PageRank, and trust score

In [ ]:
node_features = pd.DataFrame({'node_id': list(G.nodes())})
node_features['in_degree'] = node_features['node_id'].map(dict(G.in_degree())).astype(int)
node_features['out_degree'] = node_features['node_id'].map(dict(G.out_degree())).astype(int)
node_features['degree'] = node_features['in_degree'] + node_features['out_degree']
node_features['weighted_in_degree'] = node_features['node_id'].map(dict(G.in_degree(weight='Weight'))).astype(float)
node_features['weighted_out_degree'] = node_features['node_id'].map(dict(G.out_degree(weight='Weight'))).astype(float)
node_features['weighted_degree'] = node_features['weighted_in_degree'] + node_features['weighted_out_degree']
degree_statistics = node_features[['in_degree','out_degree','degree','weighted_in_degree','weighted_out_degree','weighted_degree']].describe().T
print(degree_statistics.to_string())

# NetworkX pure-Python PageRank is used because the runtime has no SciPy backend; the edge Weight remains the weight parameter.
pagerank = _pagerank_python(G, alpha=0.85, weight='Weight', max_iter=200, tol=1e-10)
node_features['pagerank'] = node_features['node_id'].map(pagerank).astype(float)

def min_max(series):
    low, high = series.min(), series.max()
    return pd.Series(0.0,index=series.index) if low == high else (series-low)/(high-low)

node_features['degree_norm'] = min_max(node_features['degree'])
node_features['pagerank_norm'] = min_max(node_features['pagerank'])
# This is a project modelling feature based on the specified formulation, not a measure of real-world trust.
node_features['trust_score'] = 0.6*node_features['degree_norm'] + 0.4*node_features['pagerank_norm']
assert ((node_features[['degree_norm','pagerank_norm','trust_score']] >= 0) & (node_features[['degree_norm','pagerank_norm','trust_score']] <= 1)).all().all()
print(node_features[['degree_norm','pagerank_norm','trust_score']].agg(['min','max']).to_string())
save_histogram(node_features['degree'],'Induced-graph degree distribution (log1p degree)','diffusion_ready_degree_distribution.png',log1p=True)
save_histogram(node_features['pagerank'],'Induced-graph PageRank distribution (log1p value)','diffusion_ready_pagerank_distribution.png',log1p=True)
save_histogram(node_features['trust_score'],'Induced-graph trust-score distribution','diffusion_ready_trust_score_distribution.png')

## 4. Tweet/content features for overlapping users

In [ ]:
overlap_tweets = tweets.loc[tweets['User'].isin(overlap_users)].copy()
overlap_tweets['Date_parsed'] = pd.to_datetime(overlap_tweets['Date'],utc=True,errors='coerce')
if overlap_tweets['Date_parsed'].isna().any():
    raise ValueError('Unparseable tweet dates found in the overlapping research population.')

# Target-derived names use the project-level provisional interpretation from Stage 2; original Target values remain untouched.
# TextBlob scores are supporting text features and never relabel an individual tweet.
sentiments = overlap_tweets['Tweet'].astype(str).map(lambda text: TextBlob(text).sentiment)
overlap_tweets['polarity'] = sentiments.map(lambda value: value.polarity).astype(float)
overlap_tweets['subjectivity'] = sentiments.map(lambda value: value.subjectivity).astype(float)

target_distribution = overlap_tweets['Target'].value_counts().sort_index()
print('Target counts for tweets authored by overlapping users (labels preserved):')
print(target_distribution.to_string())
user_aggregates = overlap_tweets.groupby('User').agg(tweet_count=('Tweet','size'), avg_likes=('Likes','mean'), avg_retweets=('Retweets','mean'), avg_polarity=('polarity','mean'), avg_subjectivity=('subjectivity','mean'))
target_counts = pd.crosstab(overlap_tweets['User'],overlap_tweets['Target']).reindex(columns=[0.0,1.0],fill_value=0).rename(columns={0.0:'negative_tweet_count',1.0:'positive_tweet_count'})
user_content_features = user_aggregates.join(target_counts,how='left')
user_content_features['negative_ratio'] = user_content_features['negative_tweet_count']/user_content_features['tweet_count']
user_content_features['positive_ratio'] = user_content_features['positive_tweet_count']/user_content_features['tweet_count']
users_negative = int((user_content_features['negative_tweet_count'] > 0).sum())
users_positive = int((user_content_features['positive_tweet_count'] > 0).sum())
users_both = int(((user_content_features['negative_tweet_count'] > 0) & (user_content_features['positive_tweet_count'] > 0)).sum())
print(json.dumps({'users_with_target_0_tweets':users_negative,'users_with_target_1_tweets':users_positive,'users_with_both_target_labels':users_both},indent=2))

## 5. Monthly tweet and Target distributions

In [ ]:
# Month buckets are derived from UTC tweet timestamps. The network edges themselves remain static.
overlap_tweets['month'] = overlap_tweets['Date_parsed'].dt.tz_localize(None).dt.to_period('M').astype(str)
months = pd.period_range('2022-10','2023-03',freq='M').astype(str).tolist()
monthly_volume = overlap_tweets.groupby('month').size().reindex(months,fill_value=0)
monthly_targets = pd.crosstab(overlap_tweets['month'],overlap_tweets['Target']).reindex(index=months,columns=[0.0,1.0],fill_value=0)
print('Monthly tweet volume:\n' + monthly_volume.to_string())
print('\nMonthly Target-label counts:\n' + monthly_targets.to_string())
save_bar_chart(monthly_volume.index.tolist(),monthly_volume.astype(int).tolist(),'Monthly tweet volume: overlapping users','full_overlap_monthly_tweet_volume.png',color=(228,26,28))

def save_grouped_bars(labels,left,right,title,filename):
    width,height,margin = 1200,680,85
    image = Image.new('RGB',(width,height),'white')
    draw,font = ImageDraw.Draw(image),ImageFont.load_default()
    maximum = max(list(left)+list(right)) or 1
    plot_width,plot_height = width-2*margin,height-2*margin
    draw.text((margin,25),title,fill='black',font=font)
    draw.text((width-260,25),'Target 0=blue; Target 1=orange',fill='black',font=font)
    draw.line((margin,height-margin,width-margin,height-margin),fill='black',width=2)
    draw.line((margin,margin,margin,height-margin),fill='black',width=2)
    slot,bar_width = plot_width/max(len(labels),1),max(2,int(plot_width/max(len(labels),1)*0.28))
    for i,label in enumerate(labels):
        center = margin+i*slot+slot/2
        for offset,value,color in [(-bar_width,left[i],(55,126,184)),(0,right[i],(255,127,0))]:
            x0 = int(center+offset); y0 = int(height-margin-(value/maximum)*(plot_height-45))
            draw.rectangle((x0,y0,x0+bar_width,height-margin),fill=color)
        draw.text((int(center-slot/3),height-margin+8),str(label),fill='black',font=font)
    output=FIGURES_DIR/filename; image.save(output); print(f'Saved figure: {output.relative_to(PROJECT_ROOT)}')

save_grouped_bars(months,monthly_targets[0.0].tolist(),monthly_targets[1.0].tolist(),'Monthly Target-label distribution: overlapping users','full_overlap_monthly_target_distribution.png')

## 6. Diffusion-ready full node feature table

In [ ]:
diffusion_ready_features = node_features.set_index('node_id').join(user_content_features,how='left')
# Every overlap user has tweet data by definition; fills make the output robust to future input variations.
count_columns = ['tweet_count','negative_tweet_count','positive_tweet_count']
for column in count_columns:
    diffusion_ready_features[column] = diffusion_ready_features[column].fillna(0).astype(int)
for column in ['avg_likes','avg_retweets','negative_ratio','positive_ratio','avg_polarity','avg_subjectivity']:
    diffusion_ready_features[column] = diffusion_ready_features[column].fillna(0.0)
feature_columns = ['in_degree','out_degree','degree','weighted_in_degree','weighted_out_degree','weighted_degree','pagerank','degree_norm','pagerank_norm','trust_score','tweet_count','avg_likes','avg_retweets','negative_tweet_count','positive_tweet_count','negative_ratio','positive_ratio','avg_polarity','avg_subjectivity']
diffusion_ready_features = diffusion_ready_features.reset_index()[['node_id']+feature_columns].sort_values('node_id').reset_index(drop=True)
feature_path = PROCESSED_DIR / 'diffusion_ready_node_features_full.csv'
diffusion_ready_features.to_csv(feature_path,index=False)
print(f'Saved {len(diffusion_ready_features):,} full-population node features to {feature_path.relative_to(PROJECT_ROOT)}')
diffusion_ready_features.head()

## 7. Monthly node-feature panel (October 2022 – March 2023)

In [ ]:
monthly_aggregates = overlap_tweets.groupby(['User','month']).agg(tweet_count=('Tweet','size'), avg_likes=('Likes','mean'), avg_retweets=('Retweets','mean'), avg_polarity=('polarity','mean'))
monthly_label_counts = pd.crosstab([overlap_tweets['User'],overlap_tweets['month']],overlap_tweets['Target']).reindex(columns=[0.0,1.0],fill_value=0).rename(columns={0.0:'negative_count',1.0:'positive_count'})
monthly_observed = monthly_aggregates.join(monthly_label_counts,how='left')
monthly_observed.index = monthly_observed.index.set_names(['node_id','month'])

# Create a balanced six-month user panel: inactive user-months are represented explicitly with zeros.
panel_index = pd.MultiIndex.from_product([sorted(overlap_users),months],names=['node_id','month'])
monthly_features = monthly_observed.reindex(panel_index,fill_value=0).reset_index()
monthly_features['tweet_count'] = monthly_features['tweet_count'].fillna(0).astype(int)
monthly_features['negative_count'] = monthly_features['negative_count'].fillna(0).astype(int)
monthly_features['positive_count'] = monthly_features['positive_count'].fillna(0).astype(int)
for column in ['avg_likes','avg_retweets','avg_polarity']:
    monthly_features[column] = monthly_features[column].fillna(0.0)
monthly_features['negative_ratio'] = monthly_features['negative_count'].div(monthly_features['tweet_count']).fillna(0.0)
monthly_features['positive_ratio'] = monthly_features['positive_count'].div(monthly_features['tweet_count']).fillna(0.0)
monthly_feature_columns = ['node_id','month','tweet_count','avg_likes','avg_retweets','avg_polarity','negative_count','positive_count','negative_ratio','positive_ratio']
monthly_features = monthly_features[monthly_feature_columns].sort_values(['month','node_id']).reset_index(drop=True)
monthly_feature_path = PROCESSED_DIR / 'diffusion_ready_monthly_node_features_full.csv'
monthly_features.to_csv(monthly_feature_path,index=False)
assert len(monthly_features) == len(overlap_users)*len(months)
print(f'Saved balanced panel with {len(monthly_features):,} user-month rows to {monthly_feature_path.relative_to(PROJECT_ROOT)}')
print(monthly_features.groupby('month')['tweet_count'].sum().to_string())

## 8. Research-ready summary

In [ ]:
research_ready_summary = {
    **graph_summary,
    'target_0_tweet_count': int(target_distribution.get(0.0,0)), 'target_1_tweet_count': int(target_distribution.get(1.0,0)),
    'users_with_target_0_tweets': users_negative, 'users_with_target_1_tweets': users_positive, 'users_with_both_target_labels': users_both,
    'top_pagerank': float(node_features['pagerank'].max()), 'top_trust_score': float(node_features['trust_score'].max()),
    'degree_mean': float(node_features['degree'].mean()), 'degree_median': float(node_features['degree'].median()), 'degree_max': int(node_features['degree'].max()),
    'monthly_snapshot_count': len(months), 'monthly_panel_rows': len(monthly_features),
    'limitations': [
        'The graph topology is static: network_edges.csv has no interaction timestamps.',
        'Temporal variation is represented only by evolving monthly tweet-derived node features.',
        'The induced graph is sparse and disconnected, so later GNN/EvolveGCN work must account for isolates, components, and memory/runtime costs at full scale.',
        'Self-loops are retained for analysis and require a sensitivity decision before diffusion modelling.',
        'Target-derived count names follow the project-level interpretation and preserve original labels; they do not establish an external official label definition.'
    ]
}
print(json.dumps(research_ready_summary,indent=2))
print('\nThe full overlapping-user graph is suitable for negative-information diffusion research because every included node has both observed content features and network connectivity context. Its full scale preserves real interaction structure and avoids sampling artefacts, while its sparsity, disconnected components, isolates, and static-edge assumption are computational and modelling challenges for later GNN/EvolveGCN work.')
print('The interaction topology is static; the monthly node/content feature panel, not timestamped edges, provides temporal variation.')